# Whole-bottle classifier — DINOv3 ViT-S/16

Independent bottle-level retrieval classifier for ambiguity resolution. Training is staged: heads → last four blocks → fully unfrozen backbone. Checkpoint selection and early stopping use **val_unseen Recall@1**.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['MPLBACKEND'] = 'Agg'

INPUT = Path('/kaggle/input')
PROJECT = Path('/kaggle/working/VIscaNEr')
VARIANT = 'vits16'
MODEL_ID = 'facebook/dinov3-vits16-pretrain-lvd1689m'
CONFIG_NAME = 'bottle_classifier_vits16.yaml'
RUN_NAME = 'bottle_classifier_dinov3_vits16'
CODE_DATASET_SLUG = 'viscaner-bottle-classifier-code'
DATA_DATASET_SLUG = 'viscaner-bottle-classifier-data'
OPTIONAL_WEIGHTS_DATASET_SLUG = None  # Set only if you attached an offline HF snapshot dataset.
RESUME_CHECKPOINT = None
RUN_SMOKE_TEST = True

def find_unique(dataset_slug: str, filename: str) -> Path:
    matches = sorted(path for path in INPUT.rglob(filename) if path.is_file() and dataset_slug in path.parts)
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename} inside {dataset_slug}; found: {matches}')
    return matches[0]

def run_streaming(command: list[str], log_path: Path) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    environment = os.environ.copy()
    environment['PYTHONUNBUFFERED'] = '1'
    with log_path.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=PROJECT, env=environment, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        exit_code = process.wait()
    if exit_code != 0:
        raise RuntimeError(f'Process failed with exit code {exit_code}. See {log_path}')

train_script = find_unique(CODE_DATASET_SLUG, 'train_bottle_classifier.py')
code_root = train_script.parent
metadata = find_unique(DATA_DATASET_SLUG, 'crops_metadata.csv')
manifest = find_unique(DATA_DATASET_SLUG, 'bottle_images_manifest.csv')
assert metadata.parent == manifest.parent
assert (metadata.parent / 'successful').is_dir() or (metadata.parent / 'crops' / 'successful').is_dir()
crops_root = metadata.parent / 'crops' if (metadata.parent / 'crops').is_dir() else metadata.parent
refs_root = metadata.parent / 'refs'
assert refs_root.is_dir()

shutil.copytree(code_root, PROJECT, dirs_exist_ok=True, ignore=shutil.ignore_patterns('dataset-metadata.json'))
checksum_path = PROJECT / 'code_sha256.json'
assert checksum_path.is_file(), 'Update the attached code dataset: code_sha256.json is missing'
for filename, expected in json.loads(checksum_path.read_text(encoding='utf-8')).items():
    candidate = PROJECT / filename
    assert candidate.is_file(), f'Missing code file: {filename}'
    assert hashlib.sha256(candidate.read_bytes()).hexdigest() == expected, f'Code integrity failed: {filename}'

source = (PROJECT / 'train_bottle_classifier.py').read_text(encoding='utf-8')
engine = (PROJECT / 'dinov3_retrieval.py').read_text(encoding='utf-8')
required_markers = ('install_huggingface_backbone_loader', 'stage3_epochs', 'recall_at_2', 'metrics["accuracy"]', 'def print_epoch_report(')
missing = [marker for marker in required_markers if marker not in source and marker not in engine]
assert not missing, f'Attached code dataset is stale; missing markers: {missing}'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=5.6,<6', 'huggingface_hub>=0.34,<2', 'safetensors>=0.5,<1', 'scikit-learn>=1.5,<2', 'PyYAML>=6,<7'], check=True)

import pandas as pd
import torch
import yaml
from huggingface_hub import snapshot_download

assert torch.cuda.is_available(), 'Select a GPU accelerator before running this notebook'
assert {'wine_slug', 'status', 'crop_path'}.issubset(pd.read_csv(metadata, nrows=1).columns)
print('CUDA:', torch.cuda.get_device_name(0), 'GPU count:', torch.cuda.device_count())

def resolve_model_snapshot() -> Path:
    if OPTIONAL_WEIGHTS_DATASET_SLUG:
        config = find_unique(OPTIONAL_WEIGHTS_DATASET_SLUG, 'config.json')
        snapshot = config.parent
        assert (snapshot / 'model.safetensors').is_file()
        return snapshot
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    assert token, 'Create Kaggle Secret HF_TOKEN after accepting the gated model license on Hugging Face'
    destination = PROJECT / 'hf_models' / MODEL_ID.rsplit('/', 1)[-1]
    snapshot_download(repo_id=MODEL_ID, token=token, local_dir=destination)
    (destination / 'viscaner_model_source.json').write_text(json.dumps({'model_id': MODEL_ID}, indent=2), encoding='utf-8')
    return destination

weights = resolve_model_snapshot()
config_path = PROJECT / 'configs' / CONFIG_NAME
cfg = yaml.safe_load(config_path.read_text(encoding='utf-8'))
cfg.update(project_root=str(PROJECT), weights_path=str(weights), crops_metadata_path=str(metadata), bottle_manifest_path=str(manifest), crops_root=str(crops_root), refs_root=str(refs_root), index_path=str(PROJECT / 'datasets' / VARIANT / 'index.csv'), split_summary_path=str(PROJECT / 'datasets' / VARIANT / 'split_summary.json'), models_dir=str(PROJECT / 'models' / RUN_NAME), runs_dir=str(PROJECT / 'runs' / 'bottle_classifier'), run_name=RUN_NAME, device='cuda', amp=True)
assert cfg['stage1_epochs'] == 5 and cfg['stage2_epochs'] == 10 and cfg['stage3_epochs'] == 100
assert cfg['unfreeze_last_blocks'] == 4
assert cfg['early_stopping_split'] == 'val_unseen' and cfg['early_stopping_metric'] == 'recall_at_1'
if RESUME_CHECKPOINT is None and any(Path(cfg['models_dir']).glob('*.pt')):
    raise RuntimeError('Fresh run found checkpoints. Use a clean session, change RUN_NAME, or set RESUME_CHECKPOINT.')
config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print(json.dumps(cfg, indent=2))


## CUDA smoke test

One real update is run for each trainability stage before the long job starts.

In [ ]:
if RUN_SMOKE_TEST:
    smoke_cfg = dict(cfg)
    smoke_cfg.update(models_dir=str(PROJECT / 'smoke' / VARIANT / 'models'), runs_dir=str(PROJECT / 'smoke' / VARIANT / 'runs'), run_name=f'{RUN_NAME}_smoke')
    smoke_path = PROJECT / 'configs' / f'{VARIANT}_smoke.yaml'
    smoke_path.write_text(yaml.safe_dump(smoke_cfg, sort_keys=False), encoding='utf-8')
    run_streaming([sys.executable, '-u', str(PROJECT / 'train_bottle_classifier.py'), 'all', '--variant', VARIANT, '--config', str(smoke_path), '--quick-smoke'], PROJECT / f'{VARIANT}_smoke.log')
    print('Heads, last-four-block and full-backbone smoke updates passed.')
else:
    print('Smoke test skipped explicitly.')


## Staged training

Fresh runs initialize strictly from the official Hugging Face snapshot. Logs are streamed once and written with overwrite mode, so notebook and file output do not duplicate epochs.

In [ ]:
command = [sys.executable, '-u', str(PROJECT / 'train_bottle_classifier.py'), 'all', '--variant', VARIANT, '--config', str(config_path)]
if RESUME_CHECKPOINT is not None:
    resume_path = Path(RESUME_CHECKPOINT)
    assert resume_path.is_file(), f'Resume checkpoint not found: {resume_path}'
    command += ['--resume-checkpoint', str(resume_path)]
print('Starting 5 head + 10 last-block + up to 100 full-backbone epochs.', flush=True)
print('Checkpoint and early stopping: val_unseen Recall@1.', flush=True)
run_streaming(command, PROJECT / f'{VARIANT}_training.log')
summary_path = Path(cfg['runs_dir']) / RUN_NAME / 'run_summary.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
required_metrics = {'accuracy', 'recall_at_1', 'recall_at_2', 'recall_at_5', 'recall_at_10', 'mrr', 'median_rank', 'mean_rank', 'num_queries'}
for split, values in summary['final_metrics'].items():
    missing = required_metrics - set(values)
    assert not missing, f'{split} metrics are incomplete: {sorted(missing)}'
    assert abs(values['accuracy'] - values['recall_at_1']) < 1e-12
print(json.dumps(summary, indent=2))
print('Continuation required.' if summary.get('continuation_required') else 'Training complete.')


## Curves and retrieval errors

In [ ]:
from IPython.display import display
from PIL import Image
run_dir = Path(cfg['runs_dir']) / RUN_NAME
visuals = [run_dir / 'training_curves.png', run_dir / 'retrieval_failures.png']
for audit_name in ('final_audit', 'interim_audit'):
    visuals.extend([run_dir / audit_name / 'val_unseen_worst.png', run_dir / audit_name / 'val_unseen_rank2.png', run_dir / audit_name / 'val_seen_worst.png'])
for path in visuals:
    if path.is_file():
        print(path)
        display(Image.open(path))
